# Business Entity Resolution — Final Inference Pipeline

This notebook runs the complete end-to-end entity resolution inference pipeline to generate the official submission files:
- `output/matching_results.tsv` (Leaderboard-scored entity matches)
- `output/candidate_pairs.tsv` (Blocking candidate set)
- `final_submission/submission.csv`

### Pipeline Overview:
1. **Preprocess & Normalization:** C-speed accent stripping, Unicode NFKD, legal suffix normalization, postal & address expansion.
2. **Multi-Pass Country-Partitioned Blocking:** Inverted indices on distinctive name tokens, compound prefix tokens, number-street pairs, and postal codes with adaptive block size capping (Candidate Recall > 96.9%).
3. **Pairwise Feature Engineering:** 22 pairwise features capturing name ratios, token containment, exact name matches, street number matches/mismatches, postal code consistency, and state/city alignment.
4. **Trained Production Model:** LightGBM classifier with optimal decision threshold calibrated on held-out validation data.

In [ ]:
import sys
sys.path.append("../src")
from business_entity_resolution.pipeline import EntityResolutionPipeline

# Initialize pipeline with trained LightGBM model and calibrated threshold
pipe = EntityResolutionPipeline(
    dataset_dir="../dataset",
    output_dir="../output",
    model_type="gbdt",
    threshold=0.70
)

# Execute test inference across all test sources
pipe.run(
    train_split="train",
    test_split="test",
    save_model_path="../experiments/results/model.joblib"
)

## Submission Validation
Run official hackathon validator on the generated outputs:

In [ ]:
!python ../utils/validate_submission.py \
    --matching ../output/matching_results.tsv \
    --candidate ../output/candidate_pairs.tsv \
    --test-dir ../dataset/test